# RS ai — آزمایش موتور آواتار روی GPU ابری

**برای رایانه‌های بدون GPU مناسب:** خود برنامه روی ویندوز می‌ماند و فقط مدل ویدئو در یک سرویس GPU ابری اجرا می‌شود. این دفترچه در Google Colab یا JupyterLab یک سرور GPU قابل اجراست؛ برای ایران می‌توان از سرویس‌های GPU داخلی استفاده کرد.

**هزینه:** برای مدل واقعی معمولاً GPU لازم است و رایگان بودن، میزان حافظه و سهمیه را سرویس‌دهنده تعیین می‌کند. پیش از خرید، یک تست رایگان/اعتبار آزمایشی و مشخصات GPU را بررسی کن. برای شروع، GPU انویدیا با ۲۴GB VRAM یا بیشتر هدف معقولی است، اما اجرای موفق قطعی نیست.

**حریم خصوصی:** عکس و صدا در فضای ابری منتقل و پردازش می‌شوند. فقط فایل‌هایی را آپلود کن که با این موضوع موافقی.

## اجرا
1. فایل را در Google Colab یا JupyterLab یک سرور GPU باز کن. در Colab از Runtime → Change runtime type گزینهٔ GPU را انتخاب کن.
2. سلول ۱ را اجرا کن؛ تا وقتی GPU انویدیا تأیید نشده، مدل‌ها دانلود نمی‌شوند.
3. بستهٔ ZIP خروجی RS ai را در Colab بارگذاری کن؛ یا اگر بسته نداری، عکس و فایل صوتی را جداگانه انتخاب کن. در JupyterLab، فایل‌ها را داخل پوشهٔ \`rsai_input\` قرار بده.
4. تست اول فقط ۵ ثانیه از صدا را پردازش می‌کند. پس از موفقیت می‌توانی `TEST_SECONDS = None` بگذاری تا تمام فایل صدا استفاده شود.

دفترچهٔ آزمایشی به‌صورت خودکار به پنل ویندوزی وصل نیست؛ بسته و MP4 را به‌صورت دستی بین دو محیط جابه‌جا کن.

کد مرجع: https://github.com/Omni-Avatar/OmniAvatar

In [ ]:
# سلول ۱: بررسی امن GPU و فضای دیسک پیش از دانلود مدل‌ها
import pathlib
import shutil
import subprocess
import sys

# Detect a persistent workspace in Colab, Kaggle, or standard Jupyter.
if pathlib.Path('/kaggle/working').is_dir():
    BASE_DIR = pathlib.Path('/kaggle/working')
elif pathlib.Path('/content').is_dir():
    BASE_DIR = pathlib.Path('/content')
else:
    BASE_DIR = pathlib.Path.cwd()

print('Python:', sys.version.split()[0])
print('Workspace base:', BASE_DIR)

nvidia_smi = shutil.which('nvidia-smi')
if not nvidia_smi:
    raise RuntimeError(
        'GPU is not available in this session: nvidia-smi was not found. '
        'This is a runtime/hardware limitation, not a Python import error. '
        'Do not run cells 2–6 or download model weights until an NVIDIA GPU '
        'is actually assigned by the cloud provider.'
    )

result = subprocess.run(
    [nvidia_smi],
    text=True,
    capture_output=True,
    timeout=30
)
if result.returncode != 0:
    raise RuntimeError(
        'nvidia-smi exists but could not query the GPU. Do not download models.\n'
        + (result.stderr or result.stdout or 'No diagnostic output.')
    )

print('\n--- NVIDIA status ---')
print(result.stdout)

gpu_mem = subprocess.run(
    [
        nvidia_smi,
        '--query-gpu=memory.total',
        '--format=csv,noheader,nounits'
    ],
    text=True,
    capture_output=True,
    timeout=30
)
if gpu_mem.returncode != 0 or not gpu_mem.stdout.strip():
    raise RuntimeError(
        'Could not read GPU memory. Check the assigned accelerator before continuing.'
    )

gpu_mib_values = [
    int(line.strip())
    for line in gpu_mem.stdout.splitlines()
    if line.strip().isdigit()
]
if not gpu_mib_values:
    raise RuntimeError('No usable NVIDIA GPU memory value was reported.')

gpu_mib = max(gpu_mib_values)
free_gb = shutil.disk_usage(BASE_DIR).free / (1024 ** 3)
print(f'Largest GPU memory: {gpu_mib / 1024:.1f} GB')
print(f'Free disk space: {free_gb:.1f} GB')

if gpu_mib < 12000:
    raise RuntimeError(
        'The available GPU has less than 12 GB VRAM. Stopping before the '
        'multi-GB downloads to avoid wasting the session.'
    )
if free_gb < 25:
    raise RuntimeError(
        'Less than 25 GB free disk space is available. Free space or choose '
        'a runtime with more storage before downloading the models.'
    )

print('\nHardware check passed. Continue to cell 2 only if all checks passed.')

In [ ]:
# سلول ۲: دریافت سورس رسمی OmniAvatar و نصب وابستگی‌ها
import os
import pathlib
import shutil
import subprocess
import sys

# Match the workspace selected by cell 1.
if pathlib.Path('/kaggle/working').is_dir():
    BASE_DIR = pathlib.Path('/kaggle/working')
elif pathlib.Path('/content').is_dir():
    BASE_DIR = pathlib.Path('/content')
else:
    BASE_DIR = pathlib.Path.cwd()

WORK = BASE_DIR / 'RS-ai-cloud'
REPO = BASE_DIR / 'OmniAvatar'
WORK.mkdir(parents=True, exist_ok=True)

# Recover cleanly from an incomplete, non-git folder left by an interrupted clone.
if not (REPO / '.git').exists():
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(
        ['git', 'clone', 'https://github.com/Omni-Avatar/OmniAvatar', str(REPO)],
        check=True
    )

os.chdir(REPO)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'],
    check=True
)

# Official repository dependency pins. If this install conflicts with the
# provider's runtime, stop and keep the complete error instead of retrying blindly.
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install',
        'torch==2.4.0', 'torchvision==0.19.0', 'torchaudio==2.4.0',
        '--index-url', 'https://download.pytorch.org/whl/cu124'
    ],
    check=True
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', 'huggingface_hub[cli]'],
    check=True
)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        'PyTorch cannot access CUDA in this session. Do not download model weights '
        'until the runtime has a working NVIDIA GPU.'
    )

print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print('Repository:', REPO)
print('Workspace:', WORK)

In [ ]:
# سلول ۳: دانلود وزن‌های موردنیاز؛ ممکن است چندین گیگابایت باشد
# این مرحله را فقط در محیط ابری دارای GPU اجرا کنید.
env = os.environ.copy()
env['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'
model_dir = REPO / 'pretrained_models'
model_dir.mkdir(exist_ok=True)
downloads = [
    ['hf','download','Wan-AI/Wan2.1-T2V-1.3B','--local-dir',str(model_dir/'Wan2.1-T2V-1.3B')],
    ['hf','download','facebook/wav2vec2-base-960h','--local-dir',str(model_dir/'wav2vec2-base-960h')],
    ['hf','download','OmniAvatar/OmniAvatar-1.3B','--local-dir',str(model_dir/'OmniAvatar-1.3B')],
]
for cmd in downloads:
    print('\nRunning:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
print('\nDownload step completed. Disk usage:')
subprocess.run(['du','-sh',str(model_dir)], check=False)

# Stop before inference if a large model download was incomplete.
expected = [
    model_dir / 'Wan2.1-T2V-1.3B' / 'diffusion_pytorch_model.safetensors',
    model_dir / 'Wan2.1-T2V-1.3B' / 'models_t5_umt5-xxl-enc-bf16.pth',
    model_dir / 'Wan2.1-T2V-1.3B' / 'Wan2.1_VAE.pth',
    model_dir / 'OmniAvatar-1.3B' / 'pytorch_model.pt',
    model_dir / 'wav2vec2-base-960h' / 'config.json',
]
missing = [str(item) for item in expected if not item.is_file()]
if missing:
    raise FileNotFoundError('Model files missing after download; retry the download cell before inference: ' + ', '.join(missing))

In [ ]:
# سلول ۴: دریافت بستهٔ RS ai یا انتخاب عکس و صدا
import pathlib
import shutil
import zipfile
import subprocess
from IPython.display import Video, display, FileLink

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp'}
AUDIO_EXTS = {'.wav', '.mp3', '.m4a', '.aac', '.flac', '.ogg'}

# Use the same workspace convention as cells 1–2.
if pathlib.Path('/kaggle/working').is_dir():
    BASE_DIR = pathlib.Path('/kaggle/working')
elif pathlib.Path('/content').is_dir():
    BASE_DIR = pathlib.Path('/content')
else:
    BASE_DIR = pathlib.Path.cwd()

WORK = BASE_DIR / 'RS-ai-cloud'
REPO = BASE_DIR / 'OmniAvatar'
WORK.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files as colab_files
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

IS_KAGGLE = pathlib.Path('/kaggle/input').is_dir()
input_files = []

# Colab upload dialog. If the dialog is cancelled or blocked, search existing
# runtime files rather than immediately failing.
if IS_COLAB:
    try:
        uploaded = colab_files.upload()
        if uploaded:
            input_files = [pathlib.Path.cwd() / name for name in uploaded.keys()]
    except Exception as exc:
        print('Colab upload dialog was unavailable:', exc)

# Kaggle input datasets are read-only under /kaggle/input. Add the ZIP as a
# Notebook Input first; extraction and generated files go under /kaggle/working.
if not input_files:
    search_roots = [BASE_DIR]
    if IS_COLAB:
        search_roots.append(pathlib.Path('/content'))
    if IS_KAGGLE:
        search_roots.append(pathlib.Path('/kaggle/input'))

    zip_candidates = []
    direct_candidates = []
    seen = set()

    for root_dir in search_roots:
        if not root_dir.exists():
            continue
        for p in root_dir.rglob('*'):
            if not p.is_file():
                continue
            try:
                resolved = p.resolve()
            except Exception:
                resolved = p
            if resolved in seen:
                continue
            seen.add(resolved)

            # Exclude model caches and the extracted copy from input discovery.
            if any(part in {'uploaded_bundle', '.git', 'demo_out'} for part in p.parts):
                continue
            if p.suffix.lower() == '.zip':
                zip_candidates.append(p)
            elif p.suffix.lower() in IMAGE_EXTS | AUDIO_EXTS:
                direct_candidates.append(p)

    preferred_zips = [
        p for p in zip_candidates
        if 'rs-ai-cloud-package' in p.name.lower()
        or 'rs-ai-cloud' in p.name.lower()
    ]

    if preferred_zips:
        input_files = [sorted(preferred_zips)[0]]
    elif len(zip_candidates) == 1:
        input_files = [zip_candidates[0]]
    else:
        input_files = direct_candidates

if not input_files:
    if IS_KAGGLE:
        raise FileNotFoundError(
            'No RS ai ZIP, reference image, or audio file was found. In Kaggle, '
            'click Add Input and add the dataset containing the RS ai package ZIP, '
            'then rerun this cell. Do not upload files into the read-only input folder.'
        )
    raise FileNotFoundError(
        'No RS ai ZIP, image, or audio was found. Upload the RS ai ZIP through the '
        'Colab upload button, or place an image and an audio file in the workspace.'
    )

zip_files = sorted(
    p for p in input_files
    if p.is_file() and p.suffix.lower() == '.zip'
)

bundle_dir = WORK / 'uploaded_bundle'
prompt_file = None

if zip_files:
    selected_zip = zip_files[0]
    print('Selected package:', selected_zip)

    if bundle_dir.exists():
        shutil.rmtree(bundle_dir)
    bundle_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(selected_zip, 'r') as archive:
        root = bundle_dir.resolve()
        for member in archive.infolist():
            target = (bundle_dir / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError(
                    'Unsafe path detected inside the ZIP archive; extraction stopped.'
                )
        archive.extractall(bundle_dir)

    input_files = [p for p in bundle_dir.rglob('*') if p.is_file()]
    prompt_candidates = sorted(
        p for p in bundle_dir.rglob('prompt.txt') if p.is_file()
    )
    prompt_file = prompt_candidates[0] if prompt_candidates else None
    print('ZIP extraction completed.')
else:
    # For separate image/audio files, preserve the discovered direct file list.
    input_files = [p for p in input_files if p.is_file()]

image_files = sorted(
    p for p in input_files
    if p.suffix.lower() in IMAGE_EXTS
)
audio_files = sorted(
    p for p in input_files
    if p.suffix.lower() in AUDIO_EXTS
)

if not image_files:
    raise FileNotFoundError(
        'No reference image found. Include a PNG, JPG, JPEG, or WEBP image in the ZIP/input.'
    )
if not audio_files:
    raise FileNotFoundError(
        'No audio found. Include WAV, MP3, M4A, AAC, FLAC, or OGG audio in the ZIP/input.'
    )

image_in = next(
    (
        p for p in image_files
        if p.name.lower() in {'reference.png', 'reference.jpg', 'reference.jpeg', 'reference.webp'}
    ),
    image_files[0]
)
audio_in = next(
    (p for p in audio_files if p.name.lower() == 'speech_16k.wav'),
    audio_files[0]
)

job = WORK / 'input'
job.mkdir(parents=True, exist_ok=True)
image_path = job / ('reference' + image_in.suffix.lower())
audio_path = job / 'speech_16k.wav'
shutil.copy2(image_in, image_path)

ffmpeg_bin = shutil.which('ffmpeg')
if not ffmpeg_bin:
    raise FileNotFoundError(
        'FFmpeg was not found. Cell 2 must finish successfully before this cell is run.'
    )

TEST_SECONDS = 5  # First smoke test only; do not change until a 5-second test succeeds.
ffmpeg_cmd = [
    ffmpeg_bin, '-y', '-hide_banner', '-loglevel', 'error',
    '-i', str(audio_in), '-vn', '-ac', '1', '-ar', '16000'
]
if TEST_SECONDS is not None:
    ffmpeg_cmd += ['-t', str(TEST_SECONDS)]
ffmpeg_cmd.append(str(audio_path))

converted = subprocess.run(ffmpeg_cmd, capture_output=True, text=True)
if converted.returncode != 0:
    raise RuntimeError(
        'FFmpeg audio conversion failed:\n'
        + (converted.stderr[-4000:] if converted.stderr else 'No diagnostic output.')
    )
if not audio_path.is_file() or audio_path.stat().st_size == 0:
    raise RuntimeError('The converted audio file is missing or empty.')

if prompt_file and prompt_file.is_file():
    PROMPT = prompt_file.read_text(encoding='utf-8').strip()
else:
    PROMPT = (
        'Photorealistic adult presenter speaking naturally to camera; preserve the identity '
        'and appearance of the reference person; natural head movement and expressive but '
        'plausible hand and arm gestures; stable camera; coherent body motion; no text, '
        'subtitles or logos.'
    )

PROMPT = PROMPT.replace('@@', ' ').replace('\n', ' ').strip()
input_file = WORK / 'rsai_input.txt'
input_file.write_text(
    f'{PROMPT}@@{image_path}@@{audio_path}\n',
    encoding='utf-8'
)

print('\nInput preparation completed.')
print('Runtime:', 'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Jupyter/other')
print('Reference image:', image_path)
print('Prepared audio:', audio_path)
print('Test duration:', TEST_SECONDS, 'seconds')
print('Input manifest:', input_file)
print('Image bytes:', image_path.stat().st_size)
print('Audio bytes:', audio_path.stat().st_size)
print('OmniAvatar repository exists:', REPO.is_dir())

In [ ]:
# سلول ۵: اجرای آزمایشی مدل. تنظیمات کم‌حافظه و تعداد گام پایین برای اولین تست.
# این تنظیمات ممکن است کیفیت پایین‌تری بدهند؛ برای کیفیت بیشتر NUM_STEPS را به 20 یا بالاتر افزایش دهید.
# Short smoke test is intentionally fast/low quality. Full run uses the recommended 25 steps.
NUM_STEPS = 8 if TEST_SECONDS is not None else 25
MAX_TOKENS = 15000
print('Inference quality mode:', 'short smoke test' if TEST_SECONDS is not None else 'full run (25 steps)')
HYPARAMS = f'dtype=fp16,num_steps={NUM_STEPS},max_tokens={MAX_TOKENS},num_persistent_param_in_dit=0,tea_cache_l1_thresh=0.14'
cmd = ['torchrun','--standalone','--nproc_per_node=1','scripts/inference.py','--config','configs/inference_1.3B.yaml','--input_file',str(input_file),'--hp='+HYPARAMS]
print('Running inference. If it reports CUDA out-of-memory, this free GPU is not sufficient for this configuration.')
print('Command:', ' '.join(cmd))
run = subprocess.run(cmd, cwd=str(REPO), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
log_path = WORK / 'omniavatar.log'
log_path.write_text(run.stdout or '', encoding='utf-8', errors='replace')
print((run.stdout or '')[-12000:])
if run.returncode != 0:
    raise RuntimeError(f'OmniAvatar failed with code {run.returncode}. Full log: {log_path}')
outputs = sorted((REPO/'demo_out').rglob('result_000.mp4'), key=lambda p:p.stat().st_mtime, reverse=True)
if not outputs:
    raise FileNotFoundError('Inference returned successfully but result_000.mp4 was not found. Inspect the log.')
video_path = outputs[0]
print('Generated video:', video_path, f'({video_path.stat().st_size:,} bytes)')
display(Video(str(video_path), embed=True))

In [ ]:
# سلول ۶: ذخیره و دریافت MP4
download_path = WORK / 'RS-ai-avatar-cloud.mp4'
shutil.copy2(video_path, download_path)
if IS_COLAB:
    colab_files.download(str(download_path))
else:
    print('از پیوند زیر فایل ویدئو را دانلود کن:')
    display(FileLink(str(download_path)))